# Phase 0.2 — Feature Engineering & Pre-Computation

This notebook pre-computes all features that can be computed **before** any model is trained.
It loads the cleaned data from `phase0_data_sampling.ipynb` and produces feature matrices
ready for consumption by Phase 1 (NCF), Phase 2 (Content-Based), and Phase 3 (Wide & Deep).

---

### Feature computation pipeline & ordering

Not all features can be pre-computed. Some depend on model outputs from earlier phases.
The full pipeline is:

```
Pre-compute (this notebook)    → Phase 1 (NCF)      → Phase 2 (BERT)        → Phase 3 (Wide & Deep)
├─ Train/test split                    │                    │                       │
├─ Static game metadata                │                    │                       │
│  (genre/category multi-hot,          │                    │                       │
│   price, metacritic, platform,       │                    │                       │
│   is_free, publisher portfolio)      │                    │                       │
├─ LOO aggregate features              │                    │                       │
│  (user_positive_ratio,               ▼                    ▼                       │
│   game_positive_ratio,         NCF user/item      Content-based                   │
│   user_avg_playtime,           embeddings (32d)    similarity scores              │
│   game_avg_playtime,           NCF pred. scores                                   │
│   developer_avg_rating,              │                    │                       │
│   playtime_vs_user_avg)              │                    │                       │
├─ BERT description embeddings         │                    │                       │
├─ User taste profiles (BERT)          │                    │                       │
├─ Negative samples + features         └────────────────────┴──── fed as features ──┘
└─ Game community sentiment
```

### What CAN be pre-computed (this notebook)

| Feature | Why it's safe to pre-compute |
|---|---|
| Train/test temporal split | Fixed once, used by everything |
| Static game metadata (genre, category, price, metacritic, platform) | Independent of any model or interaction |
| BERT embeddings of `short_description` | Editorial content, independent of user experience |
| LOO aggregates for training rows | Fixed per-row computation: exclude the specific (u,g) pair |
| LOO aggregates for test rows | Use only train data with timestamp < t — fixed once split is made |
| Negative samples + their features | Generated from train set; no (u,g) pair to exclude |
| User taste profiles from review text | BERT aggregation of user's OTHER games — fixed per split |
| Game community sentiment | BERT aggregation of OTHER users' reviews before timestamp t |

### What CANNOT be pre-computed (depends on earlier phase models)

| Feature | Why it must wait | Consumed by |
|---|---|---|
| NCF user embeddings (32-dim) | Output of Phase 1 model | Phase 3 |
| NCF item embeddings (32-dim) | Output of Phase 1 model | Phase 3 |
| NCF predicted scores | Output of Phase 1 model | Phase 4 |
| Content-based similarity scores | Output of Phase 2 computation | Phase 4 |

### Negative sample LOO nuance

For synthetic negative samples (user u, game g) where no real interaction exists:
- **User aggregates**: Use ALL of user u's training reviews (no exclusion needed — there's no real interaction to leak)
- **Game aggregates**: Use ALL of game g's training reviews (same reasoning)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. Setup — Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import os

INPUT_DIR = "/content/drive/MyDrive/processed_steam_data"
OUTPUT_DIR = "/content/drive/MyDrive/processed_steam_data"

reviews = pd.read_parquet(os.path.join(INPUT_DIR, "reviews_filtered.parquet"))
apps = pd.read_parquet(os.path.join(INPUT_DIR, "apps_filtered.parquet"))
app_genres = pd.read_parquet(os.path.join(INPUT_DIR, "app_genres_filtered.parquet"))
app_categories = pd.read_parquet(os.path.join(INPUT_DIR, "app_categories_filtered.parquet"))
app_developers = pd.read_parquet(os.path.join(INPUT_DIR, "app_developers_filtered.parquet"))
app_publishers = pd.read_parquet(os.path.join(INPUT_DIR, "app_publishers_filtered.parquet"))
app_platforms = pd.read_parquet(os.path.join(INPUT_DIR, "app_platforms_filtered.parquet"))
platforms = pd.read_parquet(os.path.join(INPUT_DIR, "platforms.parquet"))
developers = pd.read_parquet(os.path.join(INPUT_DIR, "developers.parquet"))
publishers = pd.read_parquet(os.path.join(INPUT_DIR, "publishers.parquet"))

print(f"Loaded: {len(reviews):,} reviews, {len(apps):,} apps")
print(f"Users: {reviews['author_steamid'].nunique():,}, Games: {reviews['appid'].nunique():,}")

Loaded: 224,895 reviews, 32,883 apps
Users: 36,840, Games: 32,883


## 1. Temporal Train/Test Split

Per-user split: for each user, the oldest 80% of reviews go to train, the most recent 20% go to test
(with at least 1 held-out review per user).

In [ ]:
# Sort by user and timestamp
reviews = reviews.sort_values(["author_steamid", "timestamp_created"]).reset_index(drop=True)

# Per-user split: last review per user goes to test, rest to train
# For users with >= 5 reviews, hold out the last 20%; for users with < 5, hold out the last 1
def assign_split(group):
    n = len(group)
    n_test = max(1, int(n * 0.2))  # at least 1 test sample
    split = ['train'] * (n - n_test) + ['test'] * n_test
    group['split'] = split
    return group

reviews = reviews.groupby("author_steamid", group_keys=False).apply(assign_split)

train_df = reviews[reviews["split"] == "train"].copy()
test_df = reviews[reviews["split"] == "test"].copy()

print(f"Train: {len(train_df):,} reviews ({len(train_df)/len(reviews)*100:.1f}%)")
print(f"Test:  {len(test_df):,} reviews ({len(test_df)/len(reviews)*100:.1f}%)")
print(f"\nTrain users: {train_df['author_steamid'].nunique():,}")
print(f"Test users:  {test_df['author_steamid'].nunique():,}")
print(f"\nTrain label dist: {train_df['voted_up'].value_counts().to_dict()}")
print(f"Test label dist:  {test_df['voted_up'].value_counts().to_dict()}")

Train: 173,832 reviews (77.3%)
Test:  51,063 reviews (22.7%)

Train users: 36,161
Test users:  36,840

Train label dist: {True: 129846, False: 43986}
Test label dist:  {True: 38866, False: 12197}


/tmp/ipykernel_860/145167388.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  reviews = reviews.groupby("author_steamid", group_keys=False).apply(assign_split)


## 2. Static Game Features

These are independent of any user interaction — safe to compute once.

### 2a. Genre Multi-Hot Encoding

In [ ]:
# Genre multi-hot: one column per canonical English genre
genre_pivot = app_genres.assign(val=1).pivot_table(
    index="appid", columns="genre_name", values="val", fill_value=0
)
genre_pivot.columns = [f"genre_{c}" for c in genre_pivot.columns]

print(f"Genre multi-hot: {genre_pivot.shape[0]:,} games x {genre_pivot.shape[1]} genres")
print(f"Games with genre info: {genre_pivot.shape[0]:,} / {len(apps):,}")
genre_pivot.head()

Genre multi-hot: 32,810 games x 28 genres
Games with genre info: 32,810 / 32,883


,genre_Accounting,genre_Action,genre_Adventure,genre_Animation & Modeling,genre_Audio Production,genre_Casual,genre_Design & Illustration,genre_Early Access,genre_Education,genre_Free To Play,...,genre_Racing,genre_Sexual Content,genre_Simulation,genre_Software Training,genre_Sports,genre_Strategy,genre_Utilities,genre_Video Production,genre_Violent,genre_Web Publishing
appid,,,,,,,,,,,,,,,,,,,,,
400,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1300,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1313,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1520,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1640,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


### 2b. Category Multi-Hot Encoding

In [ ]:
# Category multi-hot
cat_pivot = app_categories.assign(val=1).pivot_table(
    index="appid", columns="category_name", values="val", fill_value=0
)
cat_pivot.columns = [f"cat_{c}" for c in cat_pivot.columns]

print(f"Category multi-hot: {cat_pivot.shape[0]:,} games x {cat_pivot.shape[1]} categories")
cat_pivot.head()

Category multi-hot: 32,717 games x 58 categories


,cat_Adjustable Difficulty,cat_Adjustable Text Size,cat_Camera Comfort,cat_Captions available,cat_Chat Speech-to-text,cat_Chat Text-to-speech,cat_Co-op,cat_Color Alternatives,cat_Commentary available,cat_Cross-Platform Multiplayer,...,cat_SteamVR Collectibles,cat_Stereo Sound,cat_Subtitle Options,cat_Surround Sound,cat_Touch Only Option,cat_Tracked Controller Support,cat_VR Only,cat_VR Support,cat_VR Supported,cat_Valve Anti-Cheat enabled
appid,,,,,,,,,,,,,,,,,,,,,
400,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1300,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1520,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1640,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 2c. Platform Multi-Hot Encoding

In [ ]:
# Platform multi-hot
platform_map = platforms.set_index("id")["name"].to_dict()
app_platforms_named = app_platforms.copy()
app_platforms_named["platform_name"] = app_platforms_named["platform_id"].map(platform_map)

plat_pivot = app_platforms_named.assign(val=1).pivot_table(
    index="appid", columns="platform_name", values="val", fill_value=0
)
plat_pivot.columns = [f"platform_{c}" for c in plat_pivot.columns]

print(f"Platform multi-hot: {plat_pivot.shape[0]:,} games x {plat_pivot.shape[1]} platforms")
plat_pivot.head()

Platform multi-hot: 32,883 games x 3 platforms


,platform_linux,platform_mac,platform_windows
appid,,,
400,1.0,0.0,1.0
1300,0.0,0.0,1.0
1313,0.0,0.0,1.0
1520,1.0,1.0,1.0
1640,0.0,0.0,1.0


### 2d. Scalar Game Features

In [ ]:
# Scalar features from apps table
game_scalar = apps[["appid", "is_free", "required_age", "metacritic_score",
                     "mat_final_price_log"]].copy()

# Impute metacritic_score with genre-median
# First get primary genre per game (most common genre assignment)
primary_genre = app_genres.drop_duplicates(subset="appid", keep="first").set_index("appid")["genre_name"]
game_scalar = game_scalar.merge(primary_genre.rename("primary_genre"), left_on="appid", right_index=True, how="left")

genre_median_metacritic = game_scalar.groupby("primary_genre")["metacritic_score"].transform("median")
game_scalar["metacritic_score"] = game_scalar["metacritic_score"].fillna(genre_median_metacritic)
# Fill remaining NaN with global median
global_median = game_scalar["metacritic_score"].median()
game_scalar["metacritic_score"] = game_scalar["metacritic_score"].fillna(global_median)

# Fill missing price with 0 (likely free games)
game_scalar["mat_final_price_log"] = game_scalar["mat_final_price_log"].fillna(0)

# is_free to int
game_scalar["is_free"] = game_scalar["is_free"].astype(int)

# Drop helper column
game_scalar = game_scalar.drop(columns=["primary_genre"])

print(f"Scalar game features: {game_scalar.shape}")
print(f"Missing values:\n{game_scalar.isnull().sum().to_string()}")
game_scalar.head()

Scalar game features: (32883, 5)
Missing values:
appid                  0
is_free                0
required_age           0
metacritic_score       0
mat_final_price_log    0


,appid,is_free,required_age,metacritic_score,mat_final_price_log
0,400,0,0,90.0,6.907755
1,1300,0,0,75.0,6.907755
2,1313,0,0,77.0,6.907755
3,1520,0,0,84.0,7.090077
4,1640,0,0,84.0,6.396930


### 2e. Publisher Portfolio Size

In [ ]:
# Publisher portfolio size (how many games does this game's publisher have?)
pub_portfolio = app_publishers.groupby("publisher_id")["appid"].nunique().rename("publisher_portfolio_size")
game_publisher = app_publishers.drop_duplicates(subset="appid", keep="first")  # primary publisher
game_publisher = game_publisher.merge(pub_portfolio, left_on="publisher_id", right_index=True, how="left")
game_publisher = game_publisher[["appid", "publisher_portfolio_size"]]

print(f"Publisher portfolio: {len(game_publisher):,} games")
print(game_publisher["publisher_portfolio_size"].describe().to_string())

Publisher portfolio: 32,648 games
count    32648.000000
mean        14.748622
std         34.550378
min          1.000000
25%          1.000000
50%          2.000000
75%         10.000000
max        263.000000


### 2f. Assemble Static Game Feature Matrix

In [ ]:
# Join all static game features
game_features = game_scalar.set_index("appid")
game_features = game_features.join(genre_pivot, how="left")
game_features = game_features.join(cat_pivot, how="left")
game_features = game_features.join(plat_pivot, how="left")
game_features = game_features.join(game_publisher.set_index("appid"), how="left")

# Fill NaN in multi-hot columns with 0 (games missing from junction tables)
game_features = game_features.fillna(0)

print(f"Static game feature matrix: {game_features.shape[0]:,} games x {game_features.shape[1]} features")
print(f"Missing values: {game_features.isnull().sum().sum()}")
game_features.head()

Static game feature matrix: 32,883 games x 94 features
Missing values: 0


,is_free,required_age,metacritic_score,mat_final_price_log,genre_Accounting,genre_Action,genre_Adventure,genre_Animation & Modeling,genre_Audio Production,genre_Casual,...,cat_Touch Only Option,cat_Tracked Controller Support,cat_VR Only,cat_VR Support,cat_VR Supported,cat_Valve Anti-Cheat enabled,platform_linux,platform_mac,platform_windows,publisher_portfolio_size
appid,,,,,,,,,,,,,,,,,,,,,
400,0,0,90.0,6.907755,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,5.0
1300,0,0,75.0,6.907755,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1313,0,0,77.0,6.907755,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,36.0
1520,0,0,84.0,7.090077,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,3.0
1640,0,0,84.0,6.396930,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,76.0


## 3. Leakage-Safe LOO Aggregate Features

For each (user, game) interaction in the training set, compute aggregate features
while **excluding that specific interaction** to prevent label leakage.

For test set rows, only use training data with `timestamp < t`.

In [ ]:
# --- User-level LOO aggregates (on training data) ---

# Precompute user totals from training data
user_totals = train_df.groupby("author_steamid").agg(
    user_voted_up_sum=("voted_up", "sum"),
    user_review_count=("voted_up", "count"),
    user_playtime_sum=("author_playtime_forever", "sum"),
).astype(float)

# VH- user_price_preference: precompute per-user price sum ---
# Use mat_final_price (raw USD cents) from apps table; fall back to mat_final_price_log if unavailable
if "mat_final_price" in apps.columns:
    game_price_map = apps.set_index("appid")["mat_final_price"].fillna(0).astype(float)
else:
    # Derive raw price from log: price = exp(log_price) - 1  (inverse of log1p)
    game_price_map = (np.expm1(apps.set_index("appid")["mat_final_price_log"].fillna(0))).astype(float)

train_df_price = train_df[["author_steamid", "appid"]].copy()
train_df_price["mat_final_price"] = train_df_price["appid"].map(game_price_map).fillna(0)

user_price_totals = train_df_price.groupby("author_steamid").agg(
    user_price_sum=("mat_final_price", "sum"),
    user_price_count=("mat_final_price", "count"),
).astype(float)

user_totals = user_totals.join(user_price_totals, how="left")
user_totals[["user_price_sum", "user_price_count"]] = user_totals[["user_price_sum", "user_price_count"]].fillna(0)

print(f"User totals computed for {len(user_totals):,} users")
print(f"  (includes user_price_sum/count for user_price_preference)")
user_totals.head()



User totals computed for 36,161 users
  (includes user_price_sum/count for user_price_preference)


,user_voted_up_sum,user_review_count,user_playtime_sum,user_price_sum,user_price_count
author_steamid,,,,,
76561197960268765,1.0,2.0,884.0,1798.0,2.0
76561197960272177,2.0,2.0,1534.0,2498.0,2.0
76561197960272191,3.0,6.0,104.0,3194.0,6.0
76561197960273880,17.0,17.0,18085.0,20059.0,17.0
76561197960275132,4.0,6.0,590.0,4794.0,6.0


In [ ]:
# --- Game-level LOO aggregates (on training data) ---

game_totals = train_df.groupby("appid").agg(
    game_voted_up_sum=("voted_up", "sum"),
    game_review_count=("voted_up", "count"),
    game_playtime_sum=("author_playtime_forever", "sum"),
).astype(float)

print(f"Game totals computed for {len(game_totals):,} games")
game_totals.head()

Game totals computed for 31,551 games


,game_voted_up_sum,game_review_count,game_playtime_sum
appid,,,
400,16.0,17.0,19258.0
1300,2.0,2.0,306.0
1313,0.0,1.0,110.0
1520,1.0,1.0,298.0
1640,2.0,2.0,1397.0


In [ ]:
# --- Developer-level aggregates (on training data) ---

# First compute game_positive_ratio (global, not LOO) for developer aggregation
game_pos_ratio_global = (game_totals["game_voted_up_sum"] / game_totals["game_review_count"]).rename("game_pos_ratio")

# Map game -> developer
game_dev = app_developers.drop_duplicates(subset="appid", keep="first").set_index("appid")["developer_id"]

# Compute developer avg rating across their games (from training data)
dev_ratings = game_pos_ratio_global.to_frame().join(game_dev, how="left")
dev_avg = dev_ratings.groupby("developer_id")["game_pos_ratio"].agg(["sum", "count"]).rename(
    columns={"sum": "dev_rating_sum", "count": "dev_game_count"}
)

print(f"Developer aggregates computed for {len(dev_avg):,} developers")
dev_avg.head()

Developer aggregates computed for 20,343 developers


,dev_rating_sum,dev_game_count
developer_id,,
9.0,0.714286,1
16.0,0.600000,1
18.0,0.833333,1
20.0,1.000000,1
22.0,0.800000,1


In [ ]:
def compute_loo_features(df, user_totals, game_totals, dev_avg, game_dev, is_train=True):
    """
    Compute leave-one-out aggregate features for each (user, game) row.

    For training rows: exclude the current (u,g) interaction from aggregates.
    For test rows: use full training aggregates (no exclusion needed since
                   the test interaction was never in the training totals).
    For synthetic negatives: use full training aggregates (no real interaction to exclude).
    """
    result = df[["author_steamid", "appid"]].copy()

    # Merge user totals
    result = result.merge(user_totals, left_on="author_steamid", right_index=True, how="left")
    # Merge game totals
    result = result.merge(game_totals, left_on="appid", right_index=True, how="left")

     # Look up the price of the current game for LOO price computation
    current_price = df["appid"].map(game_price_map).fillna(0).astype(float).values

    if is_train:
        # LOO: subtract current interaction's contribution
        voted_up_float = df["voted_up"].astype(float).values
        playtime_float = df["author_playtime_forever"].astype(float).values

        # User LOO
        result["user_positive_ratio"] = (
            (result["user_voted_up_sum"] - voted_up_float) /
            (result["user_review_count"] - 1).clip(lower=1)
        )
        result["user_avg_playtime"] = (
            (result["user_playtime_sum"] - playtime_float) /
            (result["user_review_count"] - 1).clip(lower=1)
        )
        result["user_review_count_loo"] = result["user_review_count"] - 1

        # user_price_preference (LOO): exclude current game's price
        result["user_price_preference"] = (
            (result["user_price_sum"] - current_price) /
            (result["user_price_count"] - 1).clip(lower=1)
        )

        # Game LOO
        result["game_positive_ratio"] = (
            (result["game_voted_up_sum"] - voted_up_float) /
            (result["game_review_count"] - 1).clip(lower=1)
        )
        result["game_avg_playtime"] = (
            (result["game_playtime_sum"] - playtime_float) /
            (result["game_review_count"] - 1).clip(lower=1)
        )
        result["game_review_count_loo"] = result["game_review_count"] - 1
    else:
        # Test / negative: use full training aggregates (no exclusion)
        result["user_positive_ratio"] = result["user_voted_up_sum"] / result["user_review_count"].clip(lower=1)
        result["user_avg_playtime"] = result["user_playtime_sum"] / result["user_review_count"].clip(lower=1)
        result["user_review_count_loo"] = result["user_review_count"]
        result["user_price_preference"] = result["user_price_sum"] / result["user_price_count"].clip(lower=1)


        result["game_positive_ratio"] = result["game_voted_up_sum"] / result["game_review_count"].clip(lower=1)
        result["game_avg_playtime"] = result["game_playtime_sum"] / result["game_review_count"].clip(lower=1)
        result["game_review_count_loo"] = result["game_review_count"]

    # Playtime vs user avg
    result["playtime_vs_user_avg"] = (
        df["author_playtime_forever"].values / result["user_avg_playtime"].clip(lower=1).values
    )

    # Developer avg rating (LOO: exclude current game from developer's average)
    result = result.merge(game_dev.rename("developer_id"), left_on="appid", right_index=True, how="left")
    result = result.merge(dev_avg, left_on="developer_id", right_index=True, how="left")

    if is_train:
        # Exclude current game's positive_ratio from developer average
        current_game_ratio = result["game_voted_up_sum"] / result["game_review_count"].clip(lower=1)
        result["developer_avg_rating"] = (
            (result["dev_rating_sum"] - current_game_ratio) /
            (result["dev_game_count"] - 1).clip(lower=1)
        )
    else:
        result["developer_avg_rating"] = result["dev_rating_sum"] / result["dev_game_count"].clip(lower=1)

    # Keep only the engineered features
    feature_cols = [
        "user_positive_ratio", "user_avg_playtime", "user_review_count_loo", "user_price_preference",
        "game_positive_ratio", "game_avg_playtime", "game_review_count_loo",
        "playtime_vs_user_avg", "developer_avg_rating",
    ]
    return result[feature_cols].fillna(0)

print("LOO feature function defined.")

LOO feature function defined.


In [ ]:
# Compute LOO features for train and test
train_loo = compute_loo_features(train_df, user_totals, game_totals, dev_avg, game_dev, is_train=True)
test_loo = compute_loo_features(test_df, user_totals, game_totals, dev_avg, game_dev, is_train=False)

print(f"Train LOO features: {train_loo.shape}")
print(f"Test LOO features:  {test_loo.shape}")
print(f"\nSample train LOO features:")
train_loo.head()

Train LOO features: (173832, 9)
Test LOO features:  (51063, 9)

Sample train LOO features:


,user_positive_ratio,user_avg_playtime,user_review_count_loo,user_price_preference,game_positive_ratio,game_avg_playtime,game_review_count_loo,playtime_vs_user_avg,developer_avg_rating
0,1.0,842.0,1.0,999.0,0.666667,45.166667,6.0,0.049881,0.000000
1,0.0,42.0,1.0,799.0,1.000000,307.666667,15.0,20.047619,0.000000
3,1.0,1032.0,1.0,1499.0,0.666667,265.555556,9.0,0.486434,0.571429
4,1.0,502.0,1.0,999.0,1.000000,193.666667,3.0,2.055777,0.000000
6,0.6,14.8,5.0,539.0,0.600000,210.300000,10.0,2.027027,0.000000


## 4. Informed Negative Sampling

Generate synthetic negative samples: for each user, sample games from genres they've
interacted with but did NOT review. These are harder, more realistic negatives.

In [ ]:
NEG_RATIO = 4  # 4 negatives per positive

# Build genre -> game set mapping
genre_game_map = app_genres.groupby("genre_name")["appid"].apply(set).to_dict()

# Build user -> genres mapping (from training data only)
train_user_games = train_df.groupby("author_steamid")["appid"].apply(set).to_dict()
user_genre_map = {}
for uid, game_ids in train_user_games.items():
    genres_for_user = set(app_genres[app_genres["appid"].isin(game_ids)]["genre_name"])
    user_genre_map[uid] = genres_for_user

# Game popularity weights (review count in training data)
game_popularity = train_df.groupby("appid").size()
all_game_ids = set(train_df["appid"].unique())

np.random.seed(42)
neg_rows = []

for uid, user_games in train_user_games.items():
    n_pos = len(user_games)
    n_neg = n_pos * NEG_RATIO

    # Candidate pool: games in user's genres but not reviewed by user
    user_genres = user_genre_map.get(uid, set())
    candidates = set()
    for genre in user_genres:
        candidates |= genre_game_map.get(genre, set())
    candidates -= user_games  # remove already-reviewed games

    if len(candidates) == 0:
        # Fallback to random games
        candidates = all_game_ids - user_games

    candidates = list(candidates)

    # Popularity-weighted sampling
    weights = np.array([game_popularity.get(g, 1) for g in candidates], dtype=float)
    weights /= weights.sum()

    n_sample = min(n_neg, len(candidates))
    sampled = np.random.choice(candidates, size=n_sample, replace=False, p=weights)

    for gid in sampled:
        neg_rows.append({"author_steamid": uid, "appid": gid})

neg_df = pd.DataFrame(neg_rows)
neg_df["voted_up"] = False
neg_df["is_synthetic_negative"] = True
neg_df["author_playtime_forever"] = 0.0  # no playtime for synthetic negatives

print(f"Generated {len(neg_df):,} synthetic negatives")
print(f"Ratio: {len(neg_df) / len(train_df):.1f} negatives per real interaction")
print(f"Unique users with negatives: {neg_df['author_steamid'].nunique():,}")
print(f"Unique games sampled: {neg_df['appid'].nunique():,}")

In [ ]:
# Compute LOO features for negative samples (no exclusion needed)
neg_loo = compute_loo_features(neg_df, user_totals, game_totals, dev_avg, game_dev, is_train=False)

print(f"Negative sample LOO features: {neg_loo.shape}")
neg_loo.head()

## 5. BERT Embeddings of `short_description`

Safe to pre-compute: `short_description` is editorial content, independent of user experience.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Prepare descriptions
desc_df = apps[["appid", "short_description"]].copy()
desc_df["short_description"] = desc_df["short_description"].fillna("").astype(str)

# Encode
print(f"Encoding {len(desc_df):,} descriptions...")
embeddings_384 = model.encode(
    desc_df["short_description"].tolist(),
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"Raw embeddings: {embeddings_384.shape}")

# PCA reduce to 128 dims
pca = PCA(n_components=128, random_state=42)
embeddings_128 = pca.fit_transform(embeddings_384)
print(f"PCA-reduced embeddings: {embeddings_128.shape}")
print(f"Variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Store as DataFrame
desc_emb_cols = [f"desc_emb_{i}" for i in range(128)]
desc_emb_df = pd.DataFrame(embeddings_128, columns=desc_emb_cols, index=desc_df["appid"])
desc_emb_df.head()

## 6. User Taste Profiles from Review Text (Leakage-Safe)

For each user, aggregate BERT embeddings of their reviews for **other** games
(excluding the target game). This captures general taste, not specific item opinion.

We pre-compute per-user average embeddings from training reviews, then for each
(u, g) pair, subtract game g's contribution (LOO).

In [ ]:
# Encode training review texts
train_texts = train_df[["author_steamid", "appid", "review_text"]].copy()
train_texts["review_text"] = train_texts["review_text"].fillna("").astype(str)

# Filter to non-empty reviews
train_texts_valid = train_texts[train_texts["review_text"].str.strip() != ""].copy()

print(f"Encoding {len(train_texts_valid):,} training review texts...")
review_embeddings = model.encode(
    train_texts_valid["review_text"].tolist(),
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"Review embeddings: {review_embeddings.shape}")

# PCA reduce to 128 using same approach
pca_review = PCA(n_components=128, random_state=42)
review_emb_128 = pca_review.fit_transform(review_embeddings)

# Compute per-user sum and count for LOO
review_emb_df = pd.DataFrame(review_emb_128, index=train_texts_valid.index)
review_emb_df["author_steamid"] = train_texts_valid["author_steamid"].values

user_taste_sum = review_emb_df.groupby("author_steamid")[list(range(128))].sum()
user_taste_count = review_emb_df.groupby("author_steamid")[0].count().rename("count")

# Average taste profile per user
user_taste_avg = user_taste_sum.div(user_taste_count, axis=0)
user_taste_avg.columns = [f"user_taste_{i}" for i in range(128)]

print(f"\nUser taste profiles: {user_taste_avg.shape[0]:,} users x {user_taste_avg.shape[1]} dims")
user_taste_avg.head()

## VH - 6b. User Genre Preference Vector (Leakage-Safe)

For each user, compute a genre preference profile by aggregating the genre multi-hot
vectors of their **positively reviewed** games, weighted by playtime.

This captures which genres a user gravitates toward and how much time they invest in each.

**Derivation**: For each user u in training data:
1. Select only positively reviewed games (voted_up=True)
2. For each such game, get its genre multi-hot vector
3. Weight each genre vector by the user's playtime on that game
4. Sum the weighted vectors and normalize by total playtime

The result is a continuous vector (one dimension per genre) representing the user's
genre preference distribution. Stored as a separate user-level feature matrix.

In [ ]:
# --- user_genre_vector: playtime-weighted genre profile from positively reviewed games ---

# Use genre_pivot (already computed in Section 2a) — columns are genre_{name}, index is appid
genre_cols = list(genre_pivot.columns)
n_genres = len(genre_cols)

# Filter training data to positive reviews only
train_positive = train_df[train_df["voted_up"] == True][["author_steamid", "appid", "author_playtime_forever"]].copy()

# Merge genre vectors for each game
train_pos_genres = train_positive.merge(
    genre_pivot, left_on="appid", right_index=True, how="left"
)

# Fill NaN genres with 0 (games without genre info)
train_pos_genres[genre_cols] = train_pos_genres[genre_cols].fillna(0)

# Weight each genre column by playtime
playtime_vals = train_pos_genres["author_playtime_forever"].values.reshape(-1, 1)
weighted_genres = train_pos_genres[genre_cols].values * playtime_vals

# Create weighted DataFrame for aggregation
weighted_df = pd.DataFrame(weighted_genres, columns=genre_cols, index=train_pos_genres.index)
weighted_df["author_steamid"] = train_pos_genres["author_steamid"].values
weighted_df["playtime"] = train_pos_genres["author_playtime_forever"].values

# Sum weighted genre vectors per user
user_genre_sum = weighted_df.groupby("author_steamid")[genre_cols].sum()
user_playtime_total = weighted_df.groupby("author_steamid")["playtime"].sum()

# Normalize by total playtime (avoid division by zero)
user_genre_vector = user_genre_sum.div(user_playtime_total.clip(lower=1), axis=0)

# Rename columns to user_genre_* for clarity
user_genre_vector.columns = [f"user_{c}" for c in genre_cols]  # e.g., user_genre_Action

print(f"User genre vectors: {user_genre_vector.shape[0]:,} users x {user_genre_vector.shape[1]} genres")
print(f"Users with positive reviews: {len(user_genre_vector):,} / {train_df['author_steamid'].nunique():,}")
print(f"\nSample user genre vector (top genres for first user):")
sample = user_genre_vector.iloc[0].sort_values(ascending=False)
print(sample[sample > 0].head(10).to_string())
user_genre_vector.head()

## 7. Save All Pre-Computed Features

In [ ]:
# Add split and is_synthetic_negative columns for downstream use
train_df["is_synthetic_negative"] = False
test_df["is_synthetic_negative"] = False

# Concatenate LOO features with their respective DataFrames
train_with_loo = pd.concat([train_df.reset_index(drop=True), train_loo.reset_index(drop=True)], axis=1)
test_with_loo = pd.concat([test_df.reset_index(drop=True), test_loo.reset_index(drop=True)], axis=1)
neg_with_loo = pd.concat([neg_df.reset_index(drop=True), neg_loo.reset_index(drop=True)], axis=1)

# Save everything
train_with_loo.to_parquet(os.path.join(OUTPUT_DIR, "train_with_features.parquet"), index=False)
test_with_loo.to_parquet(os.path.join(OUTPUT_DIR, "test_with_features.parquet"), index=False)
neg_with_loo.to_parquet(os.path.join(OUTPUT_DIR, "negatives_with_features.parquet"), index=False)
game_features.to_parquet(os.path.join(OUTPUT_DIR, "game_features_static.parquet"))
desc_emb_df.to_parquet(os.path.join(OUTPUT_DIR, "game_description_embeddings.parquet"))
user_taste_avg.to_parquet(os.path.join(OUTPUT_DIR, "user_taste_profiles.parquet"))
user_genre_vector.to_parquet(os.path.join(OUTPUT_DIR, "user_genre_vectors.parquet"))

print(f"All pre-computed features saved to {OUTPUT_DIR}/")
print()
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f:<45} {size_mb:>8.1f} MB")

## 8. Summary — What's Ready for Each Phase

After running this notebook, the following files are ready in `processed_data/`:

| File | Contents | Used by |
|---|---|---|
| `train_with_features.parquet` | Training reviews + all LOO aggregate features | Phase 1, 3 |
| `test_with_features.parquet` | Test reviews + all LOO aggregate features | Phase 1, 3 |
| `negatives_with_features.parquet` | Synthetic negatives + aggregate features | Phase 1, 3 |
| `game_features_static.parquet` | Genre/category/platform multi-hot + scalar features | Phase 2, 3 |
| `game_description_embeddings.parquet` | BERT embeddings of `short_description` (128-dim) | Phase 2, 3 |
| `user_taste_profiles.parquet` | User taste embeddings from review text (128-dim) | Phase 3 |

### User-Level Aggregated Features (VH)

| Feature | Description | Type | Derivation |
|---|---|---|---|
| `user_avg_playtime` | Mean playtime across reviewed games | Numeric (minutes) | LOO mean of `author_playtime_forever` |
| `user_positive_ratio` | Fraction of positive reviews | Numeric (0-1) | LOO count of `voted_up=True` / total reviews |
| `user_genre_vector` | Genre preference profile | Multi-hot vector | Playtime-weighted genre vectors of positively reviewed games |
| `user_price_preference` | Average price of reviewed games | Numeric (USD cents) | LOO mean `mat_final_price` of games reviewed |
| `user_review_count` | Number of reviews written | Numeric (count) | LOO count of reviews per `author_steamid` |

### Still needed from model phases:

| Feature | Produced by | Consumed by |
|---|---|---|
| NCF user/item embeddings (32-dim) | Phase 1 | Phase 3 |
| NCF predicted scores | Phase 1 | Phase 4 |
| Content-based similarity scores | Phase 2 | Phase 4 |

In [ ]:
# Final verification: shapes and dtypes
print("=" * 60)
print("PRE-COMPUTED FEATURE SUMMARY")
print("=" * 60)
print(f"\nTrain set:     {train_with_loo.shape} (includes {train_with_loo['is_synthetic_negative'].sum()} synthetic negatives)")
print(f"Test set:      {test_with_loo.shape}")
print(f"Negatives:     {neg_with_loo.shape}")
print(f"Game features: {game_features.shape}")
print(f"BERT desc:     {desc_emb_df.shape}")
print(f"User taste:    {user_taste_avg.shape}")
print(f"\nLOO feature columns: {list(train_loo.columns)}")
print(f"Static game feature columns ({game_features.shape[1]} total): {list(game_features.columns[:10])} ... + {game_features.shape[1]-10} more")